In [1]:
""""
Copyright (C) 2025 ETH Zurich, Stefan J.P. Chrin, Gregory Landrum, Sereina Riniker
"""

#!/usr/bin/env python
import os
import subprocess
import json
from filelock import FileLock  # For thread-safe log writes
from joblib import Parallel, delayed  # For parallel execution

#os.chdir("/localhome/schrin/PYCOMSIA")  # Set to where comsia.py lives

# --- Variables ---
base_dir = "../../../0_DATA/COMPOUNDS/PER_TARGET_SDF"  # Base directory containing target folders
sdf_activity = "dG"  # SDF activity parameter
comsia_script = "./pycomsia/comsia.py"  # Script to run
comsia_cmd = "conda run -n pycomsia python"  # Use conda run to ensure env is activated
n_splits = 100

output_dir = "./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS"  # Base output directory
log_path = "./"
log_file = os.path.join(log_path, "comsia_log.json")  # Log file path (JSON format)

TARGETS_TO_SKIP = ("galectin","p2","thrombin","cdk8","hif2a","hunt","eg5","jnk1")

# --- Log Functions ---
def load_log():
    try:
        with open(log_file, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return []

def save_log(log_entries):
    with open(log_file, 'w') as f:
        json.dump(log_entries, f, indent=2)

def log_exists(entry, log_entries):
    # Check uniqueness based on target, column_filter, and grid_resolution
    for e in log_entries:
        if (e["target"] == entry["target"] and 
            e["column_filter"] == entry["column_filter"] and 
            e["grid_resolution"] == entry["grid_resolution"]):
            return True
    return False

def add_log_entry(entry):
    lock = FileLock(log_file + ".lock")
    with lock:
        log_entries = load_log()
        if not log_exists(entry, log_entries):
            log_entries.append(entry)
            save_log(log_entries)

# --- Top-level function to run a single job (target + hyperparameters) ---
def run_job(target, sdf_file, cf, gr):
    entry = {
        "target": target,
        "column_filter": cf,
        "grid_resolution": gr,
        "sdf_file": sdf_file
    }
    with FileLock(log_file + ".lock"):
        current_log = load_log()
        if log_exists(entry, current_log):
            print(f"Skipping already run: {entry}")
            return
    # Create target-specific output folder
    target_output_dir = os.path.join(output_dir, target)
    os.makedirs(target_output_dir, exist_ok=True)
    # Build command (note the added space after n_splits value)
    cmd = (
        f"{comsia_cmd} {comsia_script} "
        f"--train_file {sdf_file} "
        f"--sdf_activity {sdf_activity} "
        f"--column_filter {cf} "
        f"--grid_resolution {gr} "
        f"--output_dir {target_output_dir} "  # Use target-specific folder
        f"--n_splits {n_splits} "
        f"--disable_visualization "
    )
    # Add --fields parameter for targets pde2 or p2 because no H bond donors -> field will be empty causing error
    if target in ("pde2", "p2"):
        cmd += "--fields SEH "
    print(f"Running: {cmd}")
    result = subprocess.run(cmd, shell=True, text=True)
    if result.returncode != 0:
        print(f"[ERROR] Command failed: {cmd}")
        print(f"Return code: {result.returncode}")
        print(f"STDERR:\n{result.stderr}")
        print(f"STDOUT:\n{result.stdout}")
        entry["status"] = "failed"
        entry["returncode"] = result.returncode
        entry["stderr"] = result.stderr
        entry["stdout"] = result.stdout
    else:
        entry["status"] = "success"
    add_log_entry(entry)

# --- Additional parameters to iterate ---
column_filters = [0.001]  # List of column filters
grid_resolutions = [1]   # List of grid resolutions

# --- Build job list over targets and hyperparameters ---
jobs = []
for target in sorted(os.listdir(base_dir)):
    if target in TARGETS_TO_SKIP:  # Skip targets
        continue
    target_path = os.path.join(base_dir, target)
    if os.path.isdir(target_path):
        sdf_file = os.path.join(target_path, f"{target}.sdf")
        if os.path.exists(sdf_file):
            for cf in column_filters:
                for gr in grid_resolutions:
                    jobs.append((target, sdf_file, cf, gr))
        else:
            print(f"SDF file not found in {target_path}")

# --- Run all jobs in parallel ---
Parallel(n_jobs=6)(
    delayed(run_job)(target, sdf_file, cf, gr)
    for target, sdf_file, cf, gr in jobs
)


Reading SDF file...

steric_field:
Original points: 4896
Points kept: 1950
Points filtered: 2946

electrostatic_field:
Original points: 4896
Points kept: 2429
Points filtered: 2467

hydrophobic_field:
Original points: 4896
Points kept: 2052
Points filtered: 2844

hbond_donor_field:
Original points: 4896
Points kept: 1479
Points filtered: 3417

hbond_acceptor_field:
Original points: 4896
Points kept: 2076
Points filtered: 2820
q2 = 0.8090 for component = 1
q2 = 0.8117 for component = 2
q2 = 0.7679 for component = 3
q2 = 0.7518 for component = 4
q2 = 0.7538 for component = 5
q2 = 0.7050 for component = 6
q2 = 0.6709 for component = 7
q2 = 0.6350 for component = 8
q2 = 0.5452 for component = 9
q2 = 0.5090 for component = 10
q2 = 0.4883 for component = 11
q2 = 0.4566 for component = 12
Optimal number of components based on maximum Q2: 2
CV results saved to ./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/cdk2/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


/home/schrin/MASTERS_THESIS_STEFAN_CHRIN_PAPER_RERUNZZ_DEPTH_5_FINAL_PAPER/ZZ_MA_FROM_GITLAB/TEST_COPY_FINAL_REPO_AND_COMPARE/ml_vs_fep/A_MODELS/PYC/STRAT/pycomsia/src/MolecularFieldCalculator.py:19: SyntaxWarning: invalid escape sequence '\*'
  "[#8&!$(*~N~[OD1]),#7&H0;!$([D4]);!$([D3]-\*=,:[$([#7,#8,#15,#16])])]"


Reading SDF file...

steric_field:
Original points: 3822
Points kept: 3010
Points filtered: 812

electrostatic_field:
Original points: 3822
Points kept: 3332
Points filtered: 490

hydrophobic_field:
Original points: 3822
Points kept: 3099
Points filtered: 723

hbond_donor_field:
Original points: 3822
Points kept: 576
Points filtered: 3246

hbond_acceptor_field:
Original points: 3822
Points kept: 2894
Points filtered: 928
q2 = 0.2118 for component = 1
q2 = 0.3163 for component = 2
q2 = 0.2598 for component = 3
q2 = 0.2123 for component = 4
q2 = 0.1477 for component = 5
q2 = 0.2050 for component = 6
q2 = 0.1552 for component = 7
q2 = 0.1317 for component = 8
q2 = 0.0847 for component = 9
q2 = 0.0498 for component = 10
q2 = -0.0726 for component = 11
q2 = -0.1085 for component = 12
Optimal number of components based on maximum Q2: 2
CV results saved to ./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/bace/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 6720
Points kept: 4368
Points filtered: 2352

electrostatic_field:
Original points: 6720
Points kept: 4995
Points filtered: 1725

hydrophobic_field:
Original points: 6720
Points kept: 4476
Points filtered: 2244

hbond_donor_field:
Original points: 6720
Points kept: 2136
Points filtered: 4584

hbond_acceptor_field:
Original points: 6720
Points kept: 4469
Points filtered: 2251
q2 = 0.5624 for component = 1
q2 = 0.5732 for component = 2
q2 = 0.6025 for component = 3
q2 = 0.5982 for component = 4
q2 = 0.5644 for component = 5
q2 = 0.6063 for component = 6
q2 = 0.6263 for component = 7
q2 = 0.6395 for component = 8
q2 = 0.6490 for component = 9
q2 = 0.6603 for component = 10
q2 = 0.6661 for component = 11
q2 = 0.6684 for component = 12
Optimal number of components based on maximum Q2: 12
CV results saved to ./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/cmet/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 4212
Points kept: 2479
Points filtered: 1733

electrostatic_field:
Original points: 4212
Points kept: 2869
Points filtered: 1343

hydrophobic_field:
Original points: 4212
Points kept: 2586
Points filtered: 1626
q2 = 0.4314 for component = 1
q2 = 0.3598 for component = 2
q2 = 0.2488 for component = 3
q2 = 0.1636 for component = 4
q2 = 0.0251 for component = 5
q2 = -0.3060 for component = 6
q2 = -0.4975 for component = 7
q2 = -0.8414 for component = 8
q2 = -1.2649 for component = 9
q2 = -2.5207 for component = 10
q2 = -4.2896 for component = 11
q2 = -5.1514 for component = 12
Optimal number of components based on maximum Q2: 1
CV results saved to ./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/pde2/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


...

Reading SDF file...

steric_field:
Original points: 4508
Points kept: 3463
Points filtered: 1045

electrostatic_field:
Original points: 4508
Points kept: 3910
Points filtered: 598

hydrophobic_field:
Original points: 4508
Points kept: 3543
Points filtered: 965

hbond_donor_field:
Original points: 4508
Points kept: 2137
Points filtered: 2371

hbond_acceptor_field:
Original points: 4508
Points kept: 3125
Points filtered: 1383
q2 = -0.0622 for component = 1
q2 = -0.2273 for component = 2
q2 = 0.0935 for component = 3
q2 = 0.0712 for component = 4
q2 = 0.1906 for component = 5
q2 = 0.2480 for component = 6
q2 = 0.3572 for component = 7
q2 = 0.3439 for component = 8
q2 = 0.3757 for component = 9
q2 = 0.3460 for component = 10
q2 = 0.3343 for component = 11
q2 = 0.3352 for component = 12
Optimal number of components based on maximum Q2: 9
CV results saved to ./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/tnks2/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 3920
Points kept: 1651
Points filtered: 2269

electrostatic_field:
Original points: 3920
Points kept: 2078
Points filtered: 1842

hydrophobic_field:
Original points: 3920
Points kept: 1768
Points filtered: 2152

hbond_donor_field:
Original points: 3920
Points kept: 997
Points filtered: 2923

hbond_acceptor_field:
Original points: 3920
Points kept: 1603
Points filtered: 2317
q2 = 0.1784 for component = 1
q2 = 0.2824 for component = 2
q2 = 0.4000 for component = 3
q2 = 0.3325 for component = 4
q2 = 0.3514 for component = 5
q2 = 0.2789 for component = 6
q2 = 0.3084 for component = 7
q2 = 0.1816 for component = 8
q2 = 0.1258 for component = 9
q2 = 0.0971 for component = 10
q2 = 0.1195 for component = 11
q2 = 0.0024 for component = 12
Optimal number of components based on maximum Q2: 3
CV results saved to ./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/tyk2/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 7182
Points kept: 4251
Points filtered: 2931

electrostatic_field:
Original points: 7182
Points kept: 4815
Points filtered: 2367

hydrophobic_field:
Original points: 7182
Points kept: 4336
Points filtered: 2846

hbond_donor_field:
Original points: 7182
Points kept: 1345
Points filtered: 5837

hbond_acceptor_field:
Original points: 7182
Points kept: 4314
Points filtered: 2868
q2 = -0.2463 for component = 1
q2 = -0.1501 for component = 2
q2 = -0.1461 for component = 3
q2 = -0.2861 for component = 4
q2 = -0.2107 for component = 5
q2 = -0.1849 for component = 6
q2 = -0.2332 for component = 7
q2 = -0.3374 for component = 8
q2 = -0.4133 for component = 9
q2 = -0.3646 for component = 10
q2 = -0.3186 for component = 11
q2 = -0.3910 for component = 12
Optimal number of components based on maximum Q2: 3
CV results saved to ./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/pfkfb3/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


Reading SDF file...

steric_field:
Original points: 7728
Points kept: 5040
Points filtered: 2688

electrostatic_field:
Original points: 7728
Points kept: 5825
Points filtered: 1903

hydrophobic_field:
Original points: 7728
Points kept: 5212
Points filtered: 2516

hbond_donor_field:
Original points: 7728
Points kept: 4576
Points filtered: 3152

hbond_acceptor_field:
Original points: 7728
Points kept: 5598
Points filtered: 2130
q2 = 0.3191 for component = 1
q2 = 0.4453 for component = 2
q2 = 0.5646 for component = 3
q2 = 0.6366 for component = 4
q2 = 0.6494 for component = 5
q2 = 0.5962 for component = 6
q2 = 0.5550 for component = 7
q2 = 0.5250 for component = 8
q2 = 0.5252 for component = 9
q2 = 0.5273 for component = 10
q2 = 0.5239 for component = 11
q2 = 0.5204 for component = 12
Optimal number of components based on maximum Q2: 5
CV results saved to ./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/syk/PLS_Analysis/CV_Results_res_1.0_var_0.001.json


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]